In [7]:
# ------------------------------------------------------------------
# 0.  Imports
# ------------------------------------------------------------------
from pathlib import Path             # nicer than os.path for paths
import pandas as pd
import numpy as np
import os

# ------------------------------------------------------------------
# Utility functions
# ------------------------------------------------------------------
def transform_student_data(input_file1: str, output_file: str):
    # Load the CSV files
    df1 = pd.read_csv(os.path.join(os.getcwd(), input_file1), delimiter=";")
    print("Loaded both data files")
    
    # Combine both datasets
    df = df1.copy()
    print("Datasets combined")
    
    # Remove exact duplicates
    df = remove_exact_duplicates(df)
    print("Exact duplicates removed")
    
    # Encoding categorical variables into binary values
    df['school'] = df['school'].map({'GP': 0, 'MS': 1})
    df['sex'] = df['sex'].map({'M': 0, 'F': 1})
    df['address'] = df['address'].map({'U': 0, 'R': 1})
    df['famsize'] = df['famsize'].map({'LE3': 0, 'GT3': 1})
    df['Pstatus'] = df['Pstatus'].map({'T': 0, 'A': 1})
    
    # Feature expansion for parent's job
    for job in ['teacher', 'health', 'services', 'at_home', 'other']:
        df[f'Mjob{job.capitalize()}'] = (df['Mjob'] == job).astype(int)
        df[f'Fjob{job.capitalize()}'] = (df['Fjob'] == job).astype(int)
    
    # Feature expansion for reason
    for reason in ['home', 'reputation', 'course', 'other']:
        df[f'reason{reason.capitalize()}'] = (df['reason'] == reason).astype(int)
    
    # Feature expansion for guardian
    for guardian in ['mother', 'father', 'other']:
        df[f'guardian{guardian.capitalize()}'] = (df['guardian'] == guardian).astype(int)
    
    # Encoding binary categorical variables
    binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
    for col in binary_cols:
        df[col] = df[col].map({'yes': 1, 'no': 0})
    
    # Dropping original categorical columns that were expanded
    df.drop(columns=['Mjob', 'Fjob', 'reason', 'guardian'], inplace=True)
    
    # Dropping the age column as per the provided suggestion
    df.drop(columns=['age'], inplace=True, errors='ignore')
    
    # Save transformed CSV
    df.to_csv(os.path.join(os.getcwd(), output_file), index=False)
    print(f"Processed data saved")

def transform_combined_student_data(input_file1: str, input_file2: str, output_file: str):
    # Load the CSV files
    df1 = pd.read_csv(os.path.join(os.getcwd(), input_file1), delimiter=";")
    df2 = pd.read_csv(os.path.join(os.getcwd(), input_file2), delimiter=";")
    print("Loaded both data files")
    
    # Combine both datasets
    df = pd.concat([df1, df2], ignore_index=True)
    print("Datasets combined")
    
    # Remove exact duplicates
    df = remove_exact_duplicates(df)
    print("Exact duplicates removed")
    
    # Encoding categorical variables into binary values
    df['school'] = df['school'].map({'GP': 0, 'MS': 1})
    df['sex'] = df['sex'].map({'M': 0, 'F': 1})
    df['address'] = df['address'].map({'U': 0, 'R': 1})
    df['famsize'] = df['famsize'].map({'LE3': 0, 'GT3': 1})
    df['Pstatus'] = df['Pstatus'].map({'T': 0, 'A': 1})
    
    # Feature expansion for parent's job
    for job in ['teacher', 'health', 'services', 'at_home', 'other']:
        df[f'Mjob{job.capitalize()}'] = (df['Mjob'] == job).astype(int)
        df[f'Fjob{job.capitalize()}'] = (df['Fjob'] == job).astype(int)
    
    # Feature expansion for reason
    for reason in ['home', 'reputation', 'course', 'other']:
        df[f'reason{reason.capitalize()}'] = (df['reason'] == reason).astype(int)
    
    # Feature expansion for guardian
    for guardian in ['mother', 'father', 'other']:
        df[f'guardian{guardian.capitalize()}'] = (df['guardian'] == guardian).astype(int)
    
    # Encoding binary categorical variables
    binary_cols = ['schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic']
    for col in binary_cols:
        df[col] = df[col].map({'yes': 1, 'no': 0})
    
    # Dropping original categorical columns that were expanded
    df.drop(columns=['Mjob', 'Fjob', 'reason', 'guardian'], inplace=True)
    
    # Save transformed CSV
    df.to_csv(os.path.join(os.getcwd(), output_file), index=False)
    print(f"Processed data saved")

def remove_exact_duplicates(df):
    """
    Removes duplicate entries from a CSV file based on specific columns.
    Keeps only one instance of each duplicate.

    :param csv_file: Path to the input CSV file.
    :param output_file: Path to save the cleaned CSV file.
    """
    # Define the columns to check for duplicates
    columns_to_check = [
        "school", "sex", "age", "address", "famsize", "Pstatus", "Medu", "Fedu",
        "Mjob", "Fjob", "reason", "nursery", "internet"
    ]
    # Print actual column names for debugging
    print("CSV Columns:", df.columns.tolist())

    # Trim spaces and standardize column names
    df.columns = df.columns.str.strip()

    # Check if all required columns exist
    missing_cols = [col for col in columns_to_check if col not in df.columns]
    if missing_cols:
        raise KeyError(f"Missing columns in CSV: {missing_cols}")

    # Drop duplicates based on the specified columns, keeping the first occurrence
    df_cleaned = df.drop_duplicates(subset=columns_to_check, keep='first')

    # Save the cleaned dataframe to a new CSV file
    return df_cleaned

# ------------------------------------------------------------------
# 1.  File paths and ingestion
# ------------------------------------------------------------------
ROOT            = Path.cwd()
MATH_DATA_PATH    = ROOT / "data/student_portugal/student-mat.csv"
PORTUGAL_DATA_PATH = ROOT / "data/student_portugal/student-por.csv"

DATAFRAME_PATH  = ROOT / "data/student_portugal/combined_noDup_processed.csv"
CODEBOOK_PATH   = ROOT / "data/student_portugal/portugal_codebook.csv"

#transform_combined_student_data(input_file1=MATH_DATA_PATH, input_file2=PORTUGAL_DATA_PATH, output_file=DATAFRAME_PATH)
transform_student_data(input_file1=PORTUGAL_DATA_PATH, output_file=DATAFRAME_PATH)

data_df     = pd.read_csv(DATAFRAME_PATH)
codebook_df = pd.read_csv(CODEBOOK_PATH)

print(codebook_df.columns.tolist())
print(f"Loaded {len(data_df):,} rows and {data_df.shape[1]} columns")

# Keep only codebook rows that correspond to columns actually present
codebook_df = codebook_df[codebook_df["NAME"].isin(data_df.columns)]
print(codebook_df.columns.tolist())

# ------------------------------------------------------------------
# 1.5  PRE‑CLEAN: drop constant columns & duplicate rows *before* main cleaning
# ------------------------------------------------------------------

def drop_constant_columns(df: pd.DataFrame):
    """Return DF without columns that have a single unique value (including NaNs)."""
    const_cols = df.columns[df.nunique(dropna=False) <= 1]
    if const_cols.any():
        print(f"Dropping {len(const_cols)} constant columns (pre‑clean): {list(const_cols)}")
    return df.drop(columns=const_cols, errors="ignore"), const_cols

# 1.5.1  Remove constant columns
data_df, const_cols_pre = drop_constant_columns(data_df)
# Keep codebook in sync
codebook_df = codebook_df[~codebook_df["NAME"].isin(const_cols_pre)]

# 1.5.2  Remove duplicate rows (keep first occurrence)
before_dup = len(data_df)
data_df     = data_df.drop_duplicates()
after_dup   = len(data_df)
print(f"Removed {before_dup - after_dup:,} duplicate rows (pre‑clean)")
print(f"Pre‑clean result: {len(data_df):,} rows and {data_df.shape[1]} columns")

# ------------------------------------------------------------------
# NEW: trim the data to the surviving code‑book variables
# ------------------------------------------------------------------
ID_COLS   = []                      # e.g. ["CNT", "CNTSCHID"] if you still need them
keep_cols = ID_COLS + codebook_df["NAME"].tolist()

data_df   = data_df.loc[:, keep_cols]

# ------------------------------------------------------------------
# 2.  Column‑level filtering (USE == 0 ➜ drop)
# ------------------------------------------------------------------
drop_cols    = codebook_df.loc[codebook_df["USE"] == 0, "NAME"]
data_df      = data_df.drop(columns=drop_cols, errors="ignore")          # returns a *new* DF
codebook_df  = codebook_df[codebook_df["USE"] != 0]                      # keep in sync

print(f"Transformed {len(data_df):,} rows and {data_df.shape[1]} columns after USE‑filter")

# ------------------------------------------------------------------
# 3.  POST‑CLEAN: drop constant columns & duplicate rows *after* main cleaning
# ------------------------------------------------------------------

data_df, const_cols_post = drop_constant_columns(data_df)
# Keep codebook in sync
codebook_df = codebook_df[~codebook_df["NAME"].isin(const_cols_post)]

before_dup_post = len(data_df)
data_df         = data_df.drop_duplicates()
after_dup_post  = len(data_df)
print(f"Removed {before_dup_post - after_dup_post:,} duplicate rows (post‑clean)")
print(f"Post‑clean result: {len(data_df):,} rows and {data_df.shape[1]} columns")

# ------------------------------------------------------------------
# 5.  Persist the cleaned frame
# ------------------------------------------------------------------
RESULT_PATH = ROOT / "data/student_portugal/portugal_clean.csv"
RESULT_PATH.parent.mkdir(parents=True, exist_ok=True)

data_df.to_csv(RESULT_PATH, index=False)
print(f"Saved cleaned data")



Loaded both data files
Datasets combined
CSV Columns: ['school', 'sex', 'age', 'address', 'famsize', 'Pstatus', 'Medu', 'Fedu', 'Mjob', 'Fjob', 'reason', 'guardian', 'traveltime', 'studytime', 'failures', 'schoolsup', 'famsup', 'paid', 'activities', 'nursery', 'higher', 'internet', 'romantic', 'famrel', 'freetime', 'goout', 'Dalc', 'Walc', 'health', 'absences', 'G1', 'G2', 'G3']
Exact duplicates removed
Processed data saved
['NAME', 'USE', 'Unnamed: 2']
Loaded 637 rows and 45 columns
['NAME', 'USE', 'Unnamed: 2']
Removed 0 duplicate rows (pre‑clean)
Pre‑clean result: 637 rows and 45 columns
Transformed 637 rows and 41 columns after USE‑filter
Removed 0 duplicate rows (post‑clean)
Post‑clean result: 637 rows and 41 columns
Saved cleaned data


# Transform the data

#School name to binary GP - 0, MS - 1
#Sex to binary M - 0, F - 1
#Age has no change - Consider dropping, how do we handle different ages and hypothesis around it?
#Address to binary U - 0, R - 1
#Famsize to binary LE3 - 0, GT3 - 1
#Pstatus to binary T - 0, A - 1

#For parents jobs we do feature expansion:
#MjobTeacher Yes - 1, No - 0 (Mjob = "teacher")
#MjobHealth Yes - 1, No - 0 (Mjob = "health")
#MjobServices Yes - 1, No - 0 (Mjob = "services")
#MjobAt_home Yes - 1, No - 0 (Mjob = "at_home")
#MjobOther Yes - 1, No - 0 (Mjob = "other")

#FjobTeacher Yes - 1, No - 0 (Fjob = "teacher")
#FjobHealth Yes - 1, No - 0 (Fjob = "health")
#FjobServices Yes - 1, No - 0 (Fjob = "services")
#FjobAt_home Yes - 1, No - 0 (Fjob = "at_home")
#FjobOther Yes - 1, No - 0 (Fjob = "other")

#For reason we do feature expansion:
#reasonHome Yes - 1, No - 0 (reason = "home")
#reasonReputation Yes - 1, No - 0 (reason = "reputation")
#reasonCourse Yes - 1, No - 0 (reason = "course")
#reasonOther Yes - 1, No - 0    (reason = "other")

#For guardian we do feature expansion:
#guardianMother Yes - 1, No - 0 (guardian = "mother")
#guardianFather Yes - 1, No - 0 (guardian = "father")
#guardianOther Yes - 1, No - 0 (guardian = "other")

#traveltime has no change
#studytime has no change
#failures has no change
#schoolsup to binary Yes - 1, No - 0
#famsup to binary Yes - 1, No - 0
#paid to binary Yes - 1, No - 0
#activities to binary Yes - 1, No - 0
#nursery to binary Yes - 1, No - 0
#higher to binary Yes - 1, No - 0
#internet to binary Yes - 1, No - 0
#romantic to binary Yes - 1, No - 0

#famrel has no change
#freetime has no change
#goout has no change
#Dalc has no change
#Walc has no change
#health has no change
#absences has no change

#G1 has no change
#G2 has no change
#G3 has no change

In [9]:
from pathlib import Path             # nicer than os.path for paths
import pandas as pd
import numpy as np
import os

ROOT            = Path.cwd()
CLEAN_DATA_PATH = ROOT / "data/student_portugal/portugal_clean.csv"

data_df     = pd.read_csv(CLEAN_DATA_PATH)

percentage_ge10 = (data_df["G3"] <= 10).mean() * 100 
print(percentage_ge10)

30.141287284144425
